In [ ]:
print("\n" + "=" * 80)
print("PROBLEMAS IDENTIFICADOS Y RECOMENDACIONES")
print("=" * 80)

# Problema 1: IDs de imagen duplicados
print("\n1️⃣  IDs DE IMAGEN DUPLICADOS")
print("-" * 80)
print("   Productos 1, 2, 3 usan el MISMO ID de imagen: 6116118849.0530")
print("   ❌ PROBLEMA: Todas muestran la misma foto")
print("   ✅ SOLUCIÓN: Cada producto necesita su propio ID de imagen en Redbubble")
print("   📌 Para obtener URLs correctas:")
print("      - Ir a https://www.redbubble.com/shop/[rbSlug]/")
print("      - Extraer la URL de imagen correcta de cada producto")

# Problema 2: Formato inválido en producto 16
print("\n2️⃣  FORMATO INVÁLIDO - PRODUCTO 16 (Imán)")
print("-" * 80)
print("   URL actual: ...750x1000.2.jpg")
print("   ❌ PROBLEMA: El '.2' antes de .jpg es inválido")
print("   ✅ SOLUCIÓN: Cambiar a ...750x1000.jpg (sin el .2)")
print("   📌 URL corregida propuesta:")
print("      https://ih1.redbubble.net/image.6116118841.0530/mo,small,fridge_close,tall_portrait,750x1000.jpg")

# Problema 3: Parámetros duplicados
print("\n3️⃣  PARÁMETROS CON FORMATO INUSUAL")
print("-" * 80)
print("   Las URLs usan comas como separadores de parámetros:")
print("   Ejemplo: image.6116118840.0530/ur,mouse_pad_small_flatlay_prop,square,1000x1000.jpg")
print("   ✅ ESTO ES NORMAL en Redbubble, pero podría causar problemas si:")
print("      - Se pasan como parámetros URL (comillas faltantes)")
print("      - Se procesan con parsers estrictos")
print("   💡 RECOMENDACIÓN: Codificar las URLs si se pasan como parámetros")

print("\n" + "=" * 80)
print("RESUMEN DE ACCIONES RECOMENDADAS")
print("=" * 80)
print("""
1. Verificar productos 1-3: Obtener IDs de imagen únicos
2. Corregir producto 16: Quitar '.2' de la URL
3. Validar en navegador: Abrir cada URL directa en un navegador
4. Implementar fallback: Mostrar placeholder si la imagen no carga (ya implementado)
5. Monitorear: Verificar regularmente si Redbubble cambia sus URLs
""")

## 3. Reporte Final y Recomendaciones

In [ ]:
def check_url_status(url, timeout=5):
    """Verificar estado HTTP de una URL"""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.head(url, timeout=timeout, headers=headers, allow_redirects=True)
        return {
            'status': response.status_code,
            'content_type': response.headers.get('content-type', 'N/A'),
            'ok': response.status_code < 400,
            'error': None
        }
    except requests.exceptions.Timeout:
        return {'status': None, 'ok': False, 'error': 'TIMEOUT', 'content_type': None}
    except requests.exceptions.ConnectionError:
        return {'status': None, 'ok': False, 'error': 'CONNECTION_ERROR', 'content_type': None}
    except Exception as e:
        return {'status': None, 'ok': False, 'error': str(type(e).__name__), 'content_type': None}

# Validar URLs
print("🔍 Validando URLs...\n")
validation_results = []

for idx, row in df.iterrows():
    url = row['img']
    result = check_url_status(url)
    
    validation_results.append({
        'id': row['id'],
        'product': row['name'],
        'status': result['status'],
        'ok': '✅' if result['ok'] else '❌',
        'error': result['error'] or 'OK'
    })
    print(f"[{row['id']:2d}] {row['name']:20s} -> {result['status'] if result['status'] else 'ERROR'}: {result['error'] or 'OK'}")

validation_df = pd.DataFrame(validation_results)
print(f"\n📊 Resumen: {validation_df['ok'].value_counts().get('✅', 0)} OK, {validation_df['ok'].value_counts().get('❌', 0)} Con problemas")

## 2. Validar Disponibilidad de URLs

Realizando solicitudes HTTP HEAD a cada URL para verificar accesibilidad.

In [ ]:
# Análisis de URLs
print("=" * 80)
print("ANÁLISIS DE PROBLEMAS EN URLs")
print("=" * 80)

# Identificar problemas
issues = []

for idx, row in df.iterrows():
    url = row['img']
    product_id = row['id']
    name = row['name']
    
    # Verificar formato
    if not url.startswith('https://'):
        issues.append({"id": product_id, "name": name, "issue": "❌ URL no usa HTTPS", "url": url[:60]})
    
    # Verificar .2.jpg (problema en ID 16)
    if '.2.jpg' in url:
        issues.append({"id": product_id, "name": name, "issue": "⚠️  Formato inválido: .2.jpg", "url": url[:60]})
    
    # Verificar IDs duplicados en parámetros
    if url.count('6116118849') > 0 and product_id in [1, 2, 3]:
        if product_id == 1:
            issues.append({"id": product_id, "name": name, "issue": "⚠️  IDs de imagen duplicados (1,2,3)", "url": url[:60]})

if issues:
    issues_df = pd.DataFrame(issues)
    print(f"\n🔴 Se encontraron {len(issues_df)} problemas potenciales:\n")
    print(issues_df.to_string(index=False))
else:
    print("\n✓ No se encontraron problemas obvios de formato")

## 1. Cargar datos de productos

Se cargaron 16 productos del catálogo con URLs de Redbubble.

In [ ]:
import pandas as pd
import requests
from urllib.parse import urlparse
import warnings
warnings.filterwarnings('ignore')

# Datos de productos
products_data = [
    {"id": 1, "cat": "ropa", "img": "https://ih1.redbubble.net/image.6116118849.0530/ssrco,classic_tee,mens_02,fafafa:ca443f4786,front,square_close_portrait,x1000.jpg", "name": "Camiseta Oversized", "rbSlug": "rh5j"},
    {"id": 2, "cat": "ropa", "img": "https://ih1.redbubble.net/image.6116118849.0530/ssrco,oversized_hoodie,mens_01,e8e6e1:aa8ffd9f0f,front,square_close_portrait,x1000.jpg", "name": "Hoodie Premium", "rbSlug": "ng59"},
    {"id": 3, "cat": "ropa", "img": "https://ih1.redbubble.net/image.6116118849.0530/ssrco,tank_top,mens_01,fafafa:ca443f4786,front,square_close_portrait,x1000.jpg", "name": "Tank Top", "rbSlug": "5xql"},
    {"id": 4, "cat": "ropa", "img": "https://ih1.redbubble.net/image.6116118842.0530/ssrco,bucket_hat,product,e5d6c5:f62bbf65ee,srp,square,1000x1000-bg,f8f8f8.jpg", "name": "Bucket Hat", "rbSlug": "3vy7"},
    {"id": 5, "cat": "ropa", "img": "https://ih1.redbubble.net/image.6116118851.0530/ur,apron_realistic_flatlay,tall_portrait,750x1000.jpg", "name": "Delantal", "rbSlug": "2b5j"},
    {"id": 6, "cat": "ropa", "img": "https://ih1.redbubble.net/image.6116118857.0530/ur,socks_flatlay_01,tall_portrait,750x1000.jpg", "name": "Calcetines", "rbSlug": "sr4q"},
    {"id": 7, "cat": "accesorios", "img": "https://ih1.redbubble.net/image.6116118840.0530/ur,mouse_pad_small_flatlay_prop,square,1000x1000.jpg", "name": "Mouse Pad", "rbSlug": "2mh5"},
    {"id": 8, "cat": "accesorios", "img": "https://ih1.redbubble.net/image.6116118838.0530/icr,iphone_17_toughmagsafe,back,a,x1000-pad,1000x1000,f8f8f8.jpg", "name": "iPhone Case", "rbSlug": "c4kx"},
    {"id": 9, "cat": "accesorios", "img": "https://ih1.redbubble.net/image.6116118865.0530/ur,desk_mat_flatlay_prop,square,1000x1000.jpg", "name": "Desk Mat", "rbSlug": "4qvw"},
    {"id": 10, "cat": "accesorios", "img": "https://ih1.redbubble.net/image.6116118841.0530/st,small,845x845-pad,1000x1000,f8f8f8.jpg", "name": "Sticker", "rbSlug": "7sgk"},
    {"id": 11, "cat": "accesorios", "img": "https://ih1.redbubble.net/image.6116118845.0530/tb,1000x1000,medium-pad,750x1000,f8f8f8.jpg", "name": "Tote Bag", "rbSlug": "xd4m"},
    {"id": 12, "cat": "accesorios", "img": "https://ih1.redbubble.net/image.6116118836.0530/ur,pin_small_front,wide_portrait,750x1000.jpg", "name": "Pin", "rbSlug": "lw9c"},
    {"id": 13, "cat": "accesorios", "img": "https://ih1.redbubble.net/image.6116118834.0530/ur,mug_lifestyle,tall_portrait,750x1000.jpg", "name": "Taza", "rbSlug": "7yqg"},
    {"id": 14, "cat": "accesorios", "img": "https://ih1.redbubble.net/image.6116118846.0530/pr,150x100,750x1000-bg,f8f8f8.jpg", "name": "Zipper Pouch", "rbSlug": "lwbc"},
    {"id": 15, "cat": "accesorios", "img": "https://ih1.redbubble.net/image.6116118839.0530/sn,x1000-pad,750x1000,f8f8f8.jpg", "name": "Notebook", "rbSlug": "qhru"},
    {"id": 16, "cat": "accesorios", "img": "https://ih1.redbubble.net/image.6116118841.0530/mo,small,fridge_close,tall_portrait,750x1000.2.jpg", "name": "Imán", "rbSlug": "9gzh"},
]

# Crear DataFrame
df = pd.DataFrame(products_data)
print(f"✓ Cargados {len(df)} productos")
print(f"\nDatos de muestra:")
print(df.head())

# Validación de URLs de Imágenes - Catálogo VerdeMeta

Verificación exhaustiva de las URLs de imágenes de productos en la tienda en línea.